In [1]:
import json
import time
import numpy as np
import pandas as pd
import urllib.request
import urllib.error
from datasets import load_dataset
import requests
import pandas as pd
import re
import tempfile
import subprocess
import sys
import os

In [3]:
#Check that Ollama is running and available before using the Qwen model, must be installed manually
OLLAMA_URL = "http://localhost:11434/api/generate"

try:
    response = requests.get(OLLAMA_URL, timeout=5)
    print("Ollama is running.")
except requests.RequestException:
    print("Ollama is not running. Please install and start Ollama.")

Ollama is running.


In [5]:
MODEL = "qwen2.5-coder:7b"

In [7]:
def call_agent(prompt, seed):
    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0,
                    "seed": seed
                }
            },
            timeout=600
        )

        response.raise_for_status()
        result = response.json()

        input_tokens = result.get("prompt_eval_count", 0)
        output_tokens = result.get("eval_count", 0)
        total_tokens = input_tokens + output_tokens

        return result["response"]

    except Exception as e:
        print(f"Model request failed: {e}")
        return None

In [9]:
dataset_syntax = load_dataset("Harryxun/BugsInPy_data",split="train[:30]")
code_syntax = dataset_syntax["buggy"]

dataset_logical = load_dataset("Muennighoff/quixbugs",split="train[:30]")
code_logical = dataset_logical["buggy_program"]
desc_logical = dataset_logical["docstring"]

In [11]:
def extract_code(response):

    if not isinstance(response, str):
        return None

    match = re.search(
        r"```(?:python)?\s*(.*?)```",
        response,
        re.DOTALL | re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    return response.strip()


def extract_error(response):
    if not isinstance(response, str):
        print("Error: Reviewer returned no valid response.")
        return None

    match = re.search(
        r"Number of errors\s*:\s*(\d+)",
        response,
        re.IGNORECASE
    )

    return int(match.group(1)) if match else None

In [13]:
def reviewer_agent(code, desc, seed):

    prompt = f"""
You are the Reviewer Agent in a multi-agent programming system.

Your responsibility is to review the original Python code.

Examine the code for:
1. Syntax errors.
2. Logical errors.
3. Incorrect or missing functionality.
4. Bad programming practices.
5. Other issues affecting correctness.

Identify distinct errors. Do not count the same underlying error more than once.

Consider whether the code achieves the intended objective.

ORIGINAL CODE:
{code}

EXPECTED OBJECTIVE:
{desc if desc else "Not provided"}

Provide:
- A list of identified errors.
- An explanation of each error.
- The possible impact of each error.
- The total number of distinct errors.

OUTPUT FORMAT:

Number of errors: [integer]

Detailed Review:
[Explain the identified errors and their impact.]
"""

    return call_agent(prompt, seed)

In [15]:
def debugger_agent(code, desc, feedback, seed):

    prompt = f"""
You are the Debugger Agent in a multi-agent programming system.

Your responsibility is to investigate the errors in the original code.

You must:
1. Examine the original code.
2. Review the Reviewer's findings.
3. Investigate the root cause of each identified error.
4. Identify additional errors the Reviewer may have missed.
5. Explain why the errors occur.
6. Suggest how they should be corrected.

Do not rewrite the entire code.

ORIGINAL CODE:
{code}

EXPECTED OBJECTIVE:
{desc if desc else "Not provided"}

FEEDBACK:
{feedback}

OUTPUT FORMAT:

Debugging Findings:
[Explain the root causes of the errors.]

Additional Issues:
[Identify any missed errors, or write "None".]

Suggestions for Fixer:
[Provide specific recommendations for correcting the code.]
"""

    return call_agent(prompt, seed)

In [17]:
def fixer_agent(code, desc, feedback, seed):

    prompt = f"""
You are the Fixer Agent in a multi-agent programming system.

Your responsibility is to correct the original Python code.

Use the feedback from the Reviewer and Debugger to:
1. Fix identified syntax errors.
2. Fix logical errors and their root causes.
3. Correct incorrect functionality.
4. Address issues affecting the intended objective.
5. Preserve functionality that is already correct.

If Tester feedback is provided, use it to revise the code
and address the reported issues.

The final code must satisfy the expected objective.
Avoid unnecessary changes.

ORIGINAL CODE:
{code}

EXPECTED OBJECTIVE:
{desc if desc else "Not provided"}


FEEDBACK:
{feedback}

Return ONLY the complete corrected Python code.
Do not include explanations or Markdown code fences.
"""

    response = call_agent(prompt, seed)

    return extract_code(response)

In [19]:
def tester_agent(code, desc, feedback, seed):

    prompt = f"""
You are the Tester Agent in a multi-agent programming system.

Your responsibility is to test the corrected Python code.

Generate relevant Python unittest test cases to check:
1. Whether the code behaves as expected.
2. Whether the intended objective is achieved.
3. Whether the identified errors have been addressed.
4. Whether important edge cases are handled.

Use the original code and other agents' feedback to understand
what needs to be tested.

CORRECTED CODE:
{code}

EXPECTED OBJECTIVE:
{desc if desc else "Not provided"}

FEEDBACK:
{feedback}

Return ONLY executable Python unittest test code.
Do not repeat the corrected code.
Do not include unittest.main().
Do not use external packages.
"""

    try:
        # Generate test cases using the model
        test_script = call_agent(prompt, seed)

        if not test_script:
            return "Tester failed to generate test cases."

        # Create temporary Python file
        with tempfile.NamedTemporaryFile(
            mode="w",
            suffix=".py",
            encoding="utf-8",
            delete=False
        ) as file:
            file.write(test_script)
            test_path = file.name

        # Execute generated test cases
        result = subprocess.run(
            [sys.executable, test_path],
            capture_output=True,
            text=True,
            timeout=30
        )

        # Collect execution results
        test_feedback = f"""
TEST EXECUTION RESULTS:

Exit code: {result.returncode}

STANDARD OUTPUT:
{result.stdout}

ERROR OUTPUT:
{result.stderr}
"""

        return test_feedback

    except subprocess.TimeoutExpired:
        return "TEST EXECUTION RESULTS:\nTest execution timed out."

    except Exception as e:
        return f"TESTER ERROR: {str(e)}"

    finally:
        if 'test_path' in locals() and os.path.exists(test_path):
            os.remove(test_path)

In [21]:
def multi_agent_model(code, desc, seed):

    start_time = time.time()

    # Reviewer
    reviewer_feedback = reviewer_agent(
        code, desc, seed
    )

    number_of_errors = extract_error(reviewer_feedback)

    # Debugger
    debugger_feedback = debugger_agent(
        code, desc, reviewer_feedback, seed
    )

    # Initial correction
    corrected_code = fixer_agent(
        code,
        desc,
        debugger_feedback,
        seed
    )


    # Tester
    tester_feedback = tester_agent(
        corrected_code,
        desc,
        debugger_feedback,
        seed
    )

    # Fixer revision if Tester reports FAIL
    test_result = re.search(
        r"Test Result\s*:\s*(PASS|FAIL)",
        tester_feedback,
        re.IGNORECASE
    )

    if not test_result or test_result.group(1).upper() == "FAIL":

        revised_code = fixer_agent(
            code,
            desc,
            tester_feedback,
            seed,
        )

        if revised_code:
            corrected_code = revised_code
    output_code = extract_code(corrected_code)

    execution_time = time.time() - start_time

    return output_code, execution_time, number_of_errors

In [ ]:
"""
examples for evaluation
seeds = [42, 123, 200]

results = []

for run, seed in enumerate(seeds, start=1):
    for index, code in enumerate(code_syntax):
        output, execution_time, error_count = multi_agent_model(
            code=code,
            desc="None",
            seed=seed
        )

        results.append({
            "run": seed,
            "example_id": index,
            "original_code": code,
            "description": "None",
            "agent_output": output,
            "execution_time": execution_time,
            "Number of errors": error_count
        })

    pd.DataFrame(results).to_csv(
        "multi_agent_results.csv",
        index=False
    )
"""

In [ ]:
"""
examples for evaluation
for run, seed in enumerate(seeds, start=1):
    for index, code in enumerate(code_logical):
        
        desc = desc_logical[index]
        
        output, execution_time, error_count = multi_agent_model(
            code=code,
            desc=desc,
            seed=seed
        )
        
        results.append({
            "run": run,
            "example_id": index,
            "original_code": code,
            "description": desc,
            "agent_output": output,
            "execution_time": execution_time,
            "Number of errors": error_count 
        })
        
    pd.DataFrame(results).to_csv(
    "multi_agent_results.csv",
    index=False
    )
"""

In [25]:
#examples results
df_mas = pd.read_csv("multi_agent_results.csv")
df_mas.head()

,run,example_id,original_code,description,agent_output,execution_time,Number of errors
0,42,0,"def _isna_new(obj):\n elif isinstance(obj, ...",NaN,"def _isna_new(obj):\n if isinstance(obj, ty...",33.527946,1.0
1,42,1,"def _isna_old(obj):\n elif isinstance(obj, ...",NaN,"def _isna_old(obj):\n if isinstance(obj, ty...",26.664691,1.0
2,42,2,"def _use_inf_as_na(key):\n globals()[""_...",NaN,"def _use_inf_as_na(key, _isna_new):\n retur...",47.110158,2.0
3,42,3,class SchemaBase(BaseModel):\n not_: Option...,NaN,"from typing import Optional, List, Any, Dict, ...",37.115653,1.0
4,42,4,class Schema(SchemaBase):\n not_: Optional[...,NaN,"from typing import List, Optional, Dict, Union...",23.882553,1.0


In [27]:
#example
code = """
def calculate_average(numbers):
    total = 0

    for number in numbers:
        total += number

    average = total / (len(numbers) - 1)

    if average > 10
        return average
    else:
        return 0


print(calculate_average([5, 10, 15, 20]))
"""
desc = "Calculate and return the average of all numbers in the list"
example = multi_agent_model(code,desc,42)

print(example)

('def calculate_average(numbers):\n    total = 0\n\n    for number in numbers:\n        total += number\n\n    average = total / len(numbers)\n\n    if average > 10:\n        return average\n    else:\n        return 0\n\n\nprint(calculate_average([5, 10, 15, 20]))', 48.11849570274353, 3)


In [43]:
code = extract_code(example[0])
print("Corrected code:\n",code)
print("\nNumber of errors:",example[2])
print("Task time:", example[1])

Corrected code:
 def calculate_average(numbers):
    total = 0

    for number in numbers:
        total += number

    average = total / len(numbers)

    if average > 10:
        return average
    else:
        return 0


print(calculate_average([5, 10, 15, 20]))

Number of errors: 3
Task time: 48.11849570274353


In [ ]:
#try the model
code = """

""" #Add your code here

desc = "" #Add a description if you want, or you can leave it empty

output = multi_agent_model(code, desc,42)

code = extract_code(output[0])
print("Corrected code:\n",code)
print("Number of errors:",output[2])
print("Task time:", output[1])